In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gym
import pandas as pd

# Hyperparameters
GAMMA = 0.99
LR = 5e-4
BUFFER_SIZE = 100000
BATCH_SIZE = 32
N_QUANTILES = 32
TARGET_UPDATE_FREQ = 1000
TAU = 0.05  # Soft update factor
EPISODES = 50
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 5000

# IQN Network
class IQN(nn.Module):
    def __init__(self, state_dim, action_dim, quantiles=32):
        super(IQN, self).__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.quantiles = quantiles
        
        self.feature_layer = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        self.quantile_layer = nn.Linear(128, 128)
        self.value_layer = nn.Linear(128, action_dim)
        
    def forward(self, state, tau):
        batch_size = state.shape[0]
        state_feature = self.feature_layer(state)
        
        tau = tau.view(batch_size, self.quantiles, 1)
        quantile_embedding = torch.cos(np.pi * tau * torch.arange(1, 129).float().to(state.device))
        quantile_embedding = self.quantile_layer(quantile_embedding).relu()
        
        x = state_feature.unsqueeze(1) + quantile_embedding
        x = self.value_layer(x)
        return x

# Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (torch.tensor(np.array(state), dtype=torch.float32),
                torch.tensor(action, dtype=torch.long),
                torch.tensor(reward, dtype=torch.float32),
                torch.tensor(np.array(next_state), dtype=torch.float32),
                torch.tensor(done, dtype=torch.float32))
    
    def __len__(self):
        return len(self.buffer)

# IQN Agent
class IQNAgent:
    def __init__(self, state_dim, action_dim):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q_net = IQN(state_dim, action_dim).to(self.device)
        self.q_target = IQN(state_dim, action_dim).to(self.device)
        self.q_target.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=LR)
        self.memory = ReplayBuffer(BUFFER_SIZE)
        self.action_dim = action_dim
    
    def select_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randint(0, self.action_dim - 1)
        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        taus = torch.rand(1, N_QUANTILES, device=self.device)
        with torch.no_grad():
            q_values = self.q_net(state, taus).mean(dim=1)
        return q_values.argmax().item()
    
    def update(self):
        if len(self.memory) < BATCH_SIZE:
            return
        state, action, reward, next_state, done = self.memory.sample(BATCH_SIZE)
        state, action, reward, next_state, done = (state.to(self.device), action.to(self.device), 
                                                   reward.to(self.device), next_state.to(self.device), 
                                                   done.to(self.device))
        
        taus = torch.rand(BATCH_SIZE, N_QUANTILES, device=self.device)
        next_taus = torch.rand(BATCH_SIZE, N_QUANTILES, device=self.device)
        
        q_values = self.q_net(state, taus).gather(2, action.view(BATCH_SIZE, 1, 1).expand(-1, N_QUANTILES, -1))
        
        with torch.no_grad():
            next_q_values = self.q_target(next_state, next_taus)
            best_actions = next_q_values.mean(dim=1).argmax(dim=1, keepdim=True)
            target_q_values = next_q_values.gather(2, best_actions.unsqueeze(1).expand(-1, N_QUANTILES, -1))
            targets = reward.unsqueeze(1) + GAMMA * (1 - done.unsqueeze(1)) * target_q_values
        
        loss = (targets - q_values).abs().mean()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
    
    def update_target_network(self):
        for target_param, param in zip(self.q_target.parameters(), self.q_net.parameters()):
            target_param.data.copy_(TAU * param.data + (1.0 - TAU) * target_param.data)

# Bitcoin Trading Environment
class BitcoinTradingEnv(gym.Env):
    def __init__(self, data):
        super(BitcoinTradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32)
    
    def reset(self):
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        return self._next_observation()
    
    def _next_observation(self):
        price = self.data.iloc[self.current_step]['Close']
        return np.array([price, self.balance, self.holdings, self.data.iloc[self.current_step]['Open'],
                         self.data.iloc[self.current_step]['High'], self.data.iloc[self.current_step]['Low']],
                        dtype=np.float32)
    
    def step(self, action):
        price = self.data.iloc[self.current_step]['Close']
        reward = 0
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        '''
        if action == 0 and self.balance >= price:
            self.holdings += 1
            self.balance -= price
        elif action == 2 and self.holdings > 0:
            self.holdings -= 1
            self.balance += price
        '''
        if action == 0 and self.balance > 0:  # 매수
            self.holdings += self.balance/price
            self.balance = 0
            #print(f'balance:{self.balance}, holdings:{self.holdings}')
        elif action == 2 and self.holdings > 0:  # 매도
            self.balance += self.holdings*price
            reward = self.holdings*price - 10000  # 이익 반영
            self.holdings = 0
            #print(f'balance:{self.balance}, holdings:{self.holdings}, reward:{reward}')
        
        if done:
            reward = self.holdings*price - self.balance
            self.balance += self.holdings*price
            self.holdings=0
            

        #print(f'balance: {self.balance}, reward: {reward}, holdings:{self.holdings}')
        
        
        return self._next_observation(),reward, done, {}

# Training Loop
data = pd.read_csv(f'/workspace/data/raw/BTCUSDT/BTCUSDT-1h-2021.csv', index_col=0)
data = data[['Open','High','Low','Close']]
env = BitcoinTradingEnv(data)
agent = IQNAgent(env.observation_space.shape[0], env.action_space.n)

epsilon = EPSILON_START
for episode in range(EPISODES):
    state = env.reset()
    done = False
    total=0
    balance=0
    while not done:
        action = agent.select_action(state, epsilon)
        next_state, reward, done, _ = env.step(action)
        #print(f'reward: {reward}')
        total += reward
        agent.memory.push(state, action, reward, next_state, done)
        agent.update()
        state = next_state
    agent.update_target_network()
    print(f"Episode {episode} Reward: {total}, Balance: {env.balance}, holdfings: {env.holdings}")


Episode 0 Reward: 8546424.082118338, Balance: 16455.052218714252, holdfings: 0
Episode 1 Reward: 3890571.6234223545, Balance: 16225.154164793192, holdfings: 0
Episode 2 Reward: -3686516.9715119153, Balance: 5146.694907942034, holdfings: 0
Episode 3 Reward: 13927166.319893228, Balance: 26814.559819098853, holdfings: 0
Episode 4 Reward: 1022484.2035689511, Balance: 9289.28749545749, holdfings: 0
Episode 5 Reward: 2724051.230925343, Balance: 9576.261946543165, holdfings: 0
Episode 6 Reward: 5279196.832084318, Balance: 14991.16329220447, holdfings: 0
Episode 7 Reward: 7580098.928115495, Balance: 14765.149185941085, holdfings: 0
Episode 8 Reward: 17990768.566481397, Balance: 23602.154842870666, holdfings: 0
Episode 9 Reward: 551034.4462646851, Balance: 8372.817868937414, holdfings: 0
Episode 10 Reward: 8022846.025393548, Balance: 14308.81709489335, holdfings: 0
Episode 11 Reward: 4830390.74869727, Balance: 11124.634683273467, holdfings: 0
Episode 12 Reward: -2047718.104250495, Balance: 7786

KeyboardInterrupt: 